# Evaluate Fine-tuned Llama 3.1 8B on Trump Speech Prediction

**Setup:** Runtime > Change runtime type > T4 GPU

This notebook:
1. Downloads the model from HuggingFace
2. Runs predictions on validation samples
3. Shows predicted vs actual side-by-side
4. Computes word overlap metrics

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes datasets

In [ ]:
# Step 2: Set your HuggingFace repo ID and token
# Replace these with your actual values

HF_REPO = "YOUR_USERNAME/llama-trump"  # <-- CHANGE THIS
HF_TOKEN = "hf_xxxx"                    # <-- CHANGE THIS (or None if public repo)

# Upload val data to Colab: click folder icon on left, upload openai_val.jsonl
# Or set the path to where you uploaded it
VAL_FILE = "openai_val.jsonl"

In [ ]:
# Step 3: Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Step 4: Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(HF_REPO, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if vram_gb >= 20:
    print("Loading in bf16 (full precision)...")
    model = AutoModelForCausalLM.from_pretrained(
        HF_REPO, dtype=torch.bfloat16, device_map="auto", token=HF_TOKEN,
    )
else:
    print("Loading in 4-bit (T4 GPU)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        HF_REPO, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN,
    )

model.eval()
print("Model loaded!")

In [ ]:
# Step 5: Load validation data
import json

with open(VAL_FILE, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(samples)} validation samples")

In [ ]:
# Step 6: Helper functions
import re
import time
import random

def generate(messages, max_new_tokens=200):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


STOP_WORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "it", "that", "this", "was", "are",
    "be", "have", "has", "had", "do", "does", "did", "will", "would",
    "could", "should", "may", "might", "shall", "can", "need", "must",
    "i", "you", "he", "she", "we", "they", "me", "him", "her", "us",
    "them", "my", "your", "his", "its", "our", "their", "what", "which",
    "who", "whom", "whose", "where", "when", "how", "not", "no", "so",
    "if", "then", "than", "too", "very", "just", "about", "up", "out",
    "all", "been", "being", "were", "am", "as", "into", "through",
    "going", "know", "like", "get", "got", "go", "said", "say",
    "also", "well", "back", "even", "still", "way", "take", "come",
    "make", "thing", "things", "think", "much", "many", "some", "any",
    "other", "over", "now", "here", "there", "these", "those", "right",
    "look", "lot", "really", "want", "tell", "people", "gonna", "dont",
    "yeah", "yes", "okay", "oh", "hey", "let", "see", "put",
}


def content_words(text):
    words = re.findall(r'[a-z]+', text.lower())
    return [w for w in words if w not in STOP_WORDS and len(w) > 2]


def word_overlap(predicted, actual):
    actual_cw = set(content_words(actual))
    if not actual_cw:
        return 0.0
    pred_cw = set(content_words(predicted))
    return len(actual_cw & pred_cw) / len(actual_cw)


print("Ready!")

In [ ]:
# Step 7: Run on random samples and see predictions
N_SAMPLES = 10  # change this to evaluate more

picks = random.sample(samples, min(N_SAMPLES, len(samples)))
overlaps = []

for i, sample in enumerate(picks):
    msgs = sample['messages']
    user_msg = next(m['content'] for m in msgs if m['role'] == 'user')
    actual = next(m['content'] for m in msgs if m['role'] == 'assistant')
    prompt_msgs = [m for m in msgs if m['role'] != 'assistant']

    # Extract event name
    event = 'Unknown'
    for line in user_msg.split('\n'):
        if line.startswith('Event:'):
            event = line.replace('Event:', '').strip()
            break

    start = time.time()
    predicted = generate(prompt_msgs)
    elapsed = time.time() - start

    overlap = word_overlap(predicted, actual)
    overlaps.append(overlap)

    print(f"\n{'='*60}")
    print(f"SAMPLE {i+1}/{len(picks)} | {event[:60]}")
    print(f"Overlap: {overlap:.1%} | Generated in {elapsed:.1f}s")
    print(f"{'='*60}")
    print(f"\nPREDICTED:\n{predicted[:400]}")
    print(f"\nACTUAL:\n{actual[:400]}")

print(f"\n\n{'='*60}")
print(f"SUMMARY: {len(overlaps)} samples")
print(f"  Avg overlap:    {sum(overlaps)/len(overlaps):.1%}")
print(f"  Median overlap: {sorted(overlaps)[len(overlaps)//2]:.1%}")
print(f"{'='*60}")

In [ ]:
# Step 8: Full evaluation (run on all val samples)
# WARNING: This takes ~20-30 min on T4 for 1,287 samples

RUN_FULL = False  # Set to True to run on all samples

if RUN_FULL:
    from collections import Counter
    all_overlaps = []
    total_hit = 0
    total_actual = 0

    for i, sample in enumerate(samples):
        msgs = sample['messages']
        actual = next(m['content'] for m in msgs if m['role'] == 'assistant')
        prompt_msgs = [m for m in msgs if m['role'] != 'assistant']

        predicted = generate(prompt_msgs)
        overlap = word_overlap(predicted, actual)
        all_overlaps.append(overlap)

        actual_cw = set(content_words(actual))
        pred_cw = set(content_words(predicted))
        total_hit += len(actual_cw & pred_cw)
        total_actual += len(actual_cw)

        if (i + 1) % 25 == 0:
            avg = sum(all_overlaps) / len(all_overlaps)
            print(f"  {i+1}/{len(samples)} | avg overlap: {avg:.1%}")

    avg = sum(all_overlaps) / len(all_overlaps)
    median = sorted(all_overlaps)[len(all_overlaps)//2]
    global_hit = total_hit / total_actual if total_actual else 0

    print(f"\n{'='*50}")
    print(f"FULL EVALUATION RESULTS")
    print(f"{'='*50}")
    print(f"  Samples:             {len(all_overlaps)}")
    print(f"  Avg word overlap:    {avg:.1%}")
    print(f"  Median word overlap: {median:.1%}")
    print(f"  Global word hit:     {global_hit:.1%}")

    buckets = Counter()
    for o in all_overlaps:
        if o < 0.1: buckets['0-10%'] += 1
        elif o < 0.2: buckets['10-20%'] += 1
        elif o < 0.3: buckets['20-30%'] += 1
        elif o < 0.4: buckets['30-40%'] += 1
        elif o < 0.5: buckets['40-50%'] += 1
        else: buckets['50%+'] += 1

    print(f"\nDistribution:")
    for b in ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50%+']:
        c = buckets.get(b, 0)
        bar = '#' * (c * 40 // len(all_overlaps)) if all_overlaps else ''
        print(f"  {b:>6}: {c:4d}  {bar}")
    print(f"{'='*50}")
else:
    print("Set RUN_FULL = True above to evaluate on all 1,287 samples")